In [56]:
import torch 
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [57]:
live = pd.read_csv("../data/samples/trial/live_metrics.csv")
verbose = pd.read_csv("../data/samples/trial/verbose_statements.csv")
initial = pd.read_csv("../data/samples/trial/initial_statement.csv")

In [58]:
live = live.drop(columns=['Unnamed: 0'])

In [59]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

gpu_util_avg = []
memory_util_avg = []
clock_util_avg = []

for i in range(1, len(reset_iters)):
    temp_metrics = live.dropna()
    gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
    memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
    clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
    gpu_util_avg.append(np.average(gpu_util_prompt))
    memory_util_avg.append(np.average(memeory_util_prompt))
    clock_util_avg.append(np.average(clock_util_prompt))

In [60]:
data = []

for (x, y, z, i) in zip(gpu_util_avg, memory_util_avg, clock_util_avg, response_time[:-1]):
    data.append([torch.Tensor([x, y, z]), torch.Tensor([i])])

In [61]:
class BenchMark(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.l1 = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU()
        )
        self.l2 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.l3 = nn.Linear(16, output_features)

    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        return x

In [62]:
model = BenchMark(3, 3)

In [63]:
def minimize_avg_time_loss(predicted, nums, ans):
    total_sum = 0
    for i in range(3):
        total_sum += (predicted[0][i]*nums[i])    
    # print("total sum")
    # print(total_sum)
    # print("total summation:")
    # print(torch.sum(total_sum))
    # print("predicted: ")
    # print(predicted)
    # print("nums: ")
    # print(nums)
    # print("label:")
    # print(ans)
    loss = nn.MSELoss()
    net_loss = loss(total_sum, ans) 
    return net_loss

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [64]:
train_data = data[:50]
test_data = data[50:]

train_dataloader = DataLoader(train_data, batch_size=10, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=10, shuffle=True)

In [65]:
model.train()

for batch, (X, y) in enumerate(train_dataloader):
    for X_mini, y_mini in zip(X, y):
        preds = model(X)
        loss = minimize_avg_time_loss(preds, X_mini, y_mini)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print("Current Loss: ")
        print(loss)

Current Loss: 
tensor(92.3857, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(399.5668, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.5647, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.0074, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.0182, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(5.8154, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(56.0681, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(6.5244, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.0001, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(6.2195, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(2.2702, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(72.5253, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(12.1230, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(23.6528, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.1743, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(6.2217, grad_fn=<MseLossBackward0>)
Current Loss: 
tensor(0.1708, grad_fn=<MseLossBackward0>)
Current

c:\Users\rahul\anaconda3\Lib\site-packages\torch\nn\modules\loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
